# Main Rerun Strategy Backtest


# Main Rerun Strategy Backtest

1. `two-stage final` 是否优于 `random_walk_zero`
2. `two-stage final` 是否优于 `macro_only RF`
3. `two-stage final` 是否优于 `single-stage RF`
4. `dense export v2 two-stage final` 在同一规则下处于什么位置

## Frozen Scope

本 notebook 固定遵守：

- holdout：`2025-07-15 ~ 2025-10-25`
- 主比较对象：
  - `two-stage final`
  - `two-stage final dense export v2`
  - `single-stage RF`
  - `macro_only RF`
  - `random_walk_zero`
- 交易规则：`long / flat`
- 主阈值：`tau = 5 bps`
- 阈值敏感性：`0 / 2.5 / 5 / 10 bps`
- 正文成本：`0 bps` 与 `5 bps`
- robustness：只比较 `mainline two-stage final` vs `dense export sensitivity v2 two-stage final`

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"/Users/horace/Coding/ML finance/project").resolve()
if not (PROJECT_ROOT / "main_rerun" / "md" / "strategy_backtest_protocol.md").exists():
    raise FileNotFoundError(f"Could not confirm project root: {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)

OUTPUT_DIR = PROJECT_ROOT / "main_rerun" / "artifacts" / "strategy_backtest"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAINLINE_STAGE2_FINAL_DIR = PROJECT_ROOT / "main_rerun" / "artifacts" / "stage2" / "final_compare"
DENSE_STAGE2_FINAL_DIR = (
    PROJECT_ROOT
    / "main_rerun"
    / "artifacts"
    / "dense_export_sensitivity"
    / "stage2"
    / "final_compare"
)
SINGLE_STAGE_DIR = (
    PROJECT_ROOT
    / "main_rerun"
    / "artifacts"
    / "ablation"
    / "one_stage_daily_raw_rich_modeling"
)

MAIN_TAU = 0.0005
TAU_VALUES = [0.0, 0.00025, 0.0005, 0.0010]
COST_BPS_VALUES = [0.0, 5.0]
ROBUSTNESS_COST_BPS = 5.0
TRADING_DAYS_PER_YEAR = 252

STRATEGY_LABEL_ORDER = [
    "two-stage final",
    "two-stage final dense export v2",
    "single-stage RF",
    "macro_only RF",
    "random_walk_zero",
]

print("Project root:", PROJECT_ROOT)
print("Output dir:", OUTPUT_DIR)
print("Main tau:", MAIN_TAU)
print("Tau values:", TAU_VALUES)
print("Cost values (bps):", COST_BPS_VALUES)

In [ ]:
def assert_exists(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    return path


def load_single_row_csv(path: Path) -> dict:
    df = pd.read_csv(assert_exists(path))
    if len(df) != 1:
        raise ValueError(f"Expected exactly one row in {path}, found {len(df)}.")
    return df.iloc[0].to_dict()


def load_selected_two_stage_predictions(base_dir: Path, source_tag: str, strategy_key: str, strategy_label: str):
    selected_row = load_single_row_csv(base_dir / "stage2_nested_cv_model_compare_selected_model.csv")
    predictions = pd.read_csv(
        assert_exists(base_dir / "stage2_nested_cv_model_compare_holdout_predictions.csv"),
        parse_dates=["date"],
    )
    matched = predictions[
        (predictions["variant_key"] == selected_row["variant_key"])
        & (predictions["model_name"] == selected_row["model_name"])
        & (predictions["spec_name"] == selected_row["spec_name"])
    ].copy()
    if matched.empty:
        raise ValueError(
            "Could not locate selected two-stage holdout predictions in "
            f"{base_dir / 'stage2_nested_cv_model_compare_holdout_predictions.csv'}."
        )
    return standardize_predictions(
        matched,
        strategy_key=strategy_key,
        strategy_label=strategy_label,
        source_tag=source_tag,
    )


def load_selected_benchmark_predictions(base_dir: Path, benchmark_key: str, strategy_key: str, strategy_label: str):
    predictions = pd.read_csv(
        assert_exists(base_dir / "stage2_nested_cv_model_compare_holdout_benchmark_predictions.csv"),
        parse_dates=["date"],
    )
    matched = predictions[predictions["benchmark_key"] == benchmark_key].copy()
    if matched.empty:
        raise ValueError(
            f"Could not locate benchmark_key={benchmark_key} in "
            f"{base_dir / 'stage2_nested_cv_model_compare_holdout_benchmark_predictions.csv'}."
        )
    matched = matched.rename(columns={"benchmark_label": "source_label"})
    return standardize_predictions(
        matched,
        strategy_key=strategy_key,
        strategy_label=strategy_label,
        source_tag=f"mainline::{benchmark_key}",
    )


def load_single_stage_predictions():
    predictions = pd.read_csv(
        assert_exists(SINGLE_STAGE_DIR / "holdout_predictions.csv"),
        parse_dates=["date"],
    )
    return standardize_predictions(
        predictions,
        strategy_key="single_stage_rf",
        strategy_label="single-stage RF",
        source_tag="mainline::single_stage_rf",
    )


def standardize_predictions(df: pd.DataFrame, strategy_key: str, strategy_label: str, source_tag: str) -> pd.DataFrame:
    required_cols = {"date", "y_true", "y_pred"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Prediction frame for {strategy_key} missing columns: {sorted(missing)}")

    out = df[["date", "y_true", "y_pred"]].copy()
    out["date"] = pd.to_datetime(out["date"])
    out["strategy_key"] = strategy_key
    out["strategy_label"] = strategy_label
    out["source_tag"] = source_tag
    out["y_true"] = out["y_true"].astype(float)
    out["y_pred"] = out["y_pred"].astype(float)
    return out.sort_values("date").reset_index(drop=True)


def validate_aligned_targets(df: pd.DataFrame):
    keys = list(df["strategy_key"].unique())
    if not keys:
        raise ValueError("No strategy rows available for alignment check.")

    reference = (
        df[df["strategy_key"] == keys[0]][["date", "y_true"]]
        .rename(columns={"y_true": "y_true_ref"})
        .sort_values("date")
        .reset_index(drop=True)
    )

    for key in keys[1:]:
        candidate = (
            df[df["strategy_key"] == key][["date", "y_true"]]
            .rename(columns={"y_true": "y_true_candidate"})
            .sort_values("date")
            .reset_index(drop=True)
        )
        merged = reference.merge(candidate, on="date", how="outer", validate="one_to_one")
        if merged["y_true_ref"].isna().any() or merged["y_true_candidate"].isna().any():
            raise ValueError(f"Target alignment failed for strategy_key={key}: date coverage mismatch.")
        if not np.allclose(merged["y_true_ref"], merged["y_true_candidate"], atol=1e-12):
            raise ValueError(f"Target alignment failed for strategy_key={key}: y_true mismatch.")


def build_mainline_strategy_input() -> pd.DataFrame:
    parts = [
        load_selected_two_stage_predictions(
            MAINLINE_STAGE2_FINAL_DIR,
            source_tag="mainline::two_stage_final",
            strategy_key="two_stage_final",
            strategy_label="two-stage final",
        ),
        load_selected_two_stage_predictions(
            DENSE_STAGE2_FINAL_DIR,
            source_tag="dense_export_v2::two_stage_final",
            strategy_key="two_stage_final_dense_export_v2",
            strategy_label="two-stage final dense export v2",
        ),
        load_single_stage_predictions(),
        load_selected_benchmark_predictions(
            MAINLINE_STAGE2_FINAL_DIR,
            benchmark_key="macro_only_random_forest",
            strategy_key="macro_only_rf",
            strategy_label="macro_only RF",
        ),
        load_selected_benchmark_predictions(
            MAINLINE_STAGE2_FINAL_DIR,
            benchmark_key="random_walk_zero",
            strategy_key="random_walk_zero",
            strategy_label="random_walk_zero",
        ),
    ]
    out = pd.concat(parts, ignore_index=True)
    validate_aligned_targets(out)
    return out.sort_values(["strategy_label", "date"]).reset_index(drop=True)


def build_two_stage_robustness_input() -> pd.DataFrame:
    parts = [
        load_selected_two_stage_predictions(
            DENSE_STAGE2_FINAL_DIR,
            source_tag="dense_export_v2::two_stage_final",
            strategy_key="two_stage_final_dense_export_v2",
            strategy_label="two-stage final dense export v2",
        ),
        load_selected_two_stage_predictions(
            MAINLINE_STAGE2_FINAL_DIR,
            source_tag="mainline::two_stage_final",
            strategy_key="two_stage_final_mainline",
            strategy_label="two-stage final",
        ),
    ]
    out = pd.concat(parts, ignore_index=True)
    validate_aligned_targets(out)
    return out.sort_values(["strategy_label", "date"]).reset_index(drop=True)


def annualized_sharpe(returns: pd.Series) -> float:
    values = returns.astype(float).to_numpy()
    if len(values) == 0:
        return float("nan")
    std = float(values.std(ddof=0))
    if std == 0.0:
        return float("nan")
    return float(np.sqrt(TRADING_DAYS_PER_YEAR) * values.mean() / std)


def max_drawdown_from_log_returns(log_returns: pd.Series) -> float:
    wealth = np.exp(log_returns.cumsum())
    running_max = wealth.cummax()
    drawdown = wealth / running_max - 1.0
    return float(drawdown.min())


def run_backtest(strategy_df: pd.DataFrame, tau: float, cost_bps: float) -> pd.DataFrame:
    out = strategy_df.sort_values("date").copy()
    out["tau"] = tau
    out["cost_bps"] = cost_bps
    out["position"] = (out["y_pred"] > tau).astype(int)
    out["prev_position"] = out["position"].shift(1).fillna(0).astype(int)
    out["turnover_event"] = (out["position"] - out["prev_position"]).abs().astype(int)
    out["entry_event"] = ((out["position"] == 1) & (out["prev_position"] == 0)).astype(int)
    out["cost_return"] = (cost_bps / 10000.0) * out["turnover_event"]
    out["gross_log_return"] = out["position"] * out["y_true"]
    out["net_log_return"] = out["gross_log_return"] - out["cost_return"]
    out["gross_cum_return"] = np.exp(out["gross_log_return"].cumsum()) - 1.0
    out["net_cum_return"] = np.exp(out["net_log_return"].cumsum()) - 1.0
    return out


def summarize_backtest(strategy_df: pd.DataFrame, tau: float, cost_bps: float) -> dict:
    bt = run_backtest(strategy_df, tau=tau, cost_bps=cost_bps)
    traded = bt[bt["position"] == 1]
    gross_final = float(bt["gross_cum_return"].iloc[-1])
    net_final = float(bt["net_cum_return"].iloc[-1])
    return {
        "strategy_key": str(bt["strategy_key"].iloc[0]),
        "strategy_label": str(bt["strategy_label"].iloc[0]),
        "source_tag": str(bt["source_tag"].iloc[0]),
        "tau": float(tau),
        "cost_bps": float(cost_bps),
        "n_obs": int(len(bt)),
        "traded_days": int(bt["position"].sum()),
        "entry_count": int(bt["entry_event"].sum()),
        "turnover": float(bt["turnover_event"].sum()),
        "gross_cumulative_return": gross_final,
        "net_cumulative_return": net_final,
        "gross_annualized_sharpe": annualized_sharpe(bt["gross_log_return"]),
        "net_annualized_sharpe": annualized_sharpe(bt["net_log_return"]),
        "gross_max_drawdown": max_drawdown_from_log_returns(bt["gross_log_return"]),
        "net_max_drawdown": max_drawdown_from_log_returns(bt["net_log_return"]),
        "hit_rate_on_traded_days": float((traded["y_true"] > 0).mean()) if len(traded) else float("nan"),
        "avg_pred_on_traded_days": float(traded["y_pred"].mean()) if len(traded) else float("nan"),
    }


def summarize_grid(all_inputs: pd.DataFrame, tau_values, cost_values) -> pd.DataFrame:
    rows = []
    for strategy_key, strategy_df in all_inputs.groupby("strategy_key", sort=False):
        for tau in tau_values:
            for cost_bps in cost_values:
                rows.append(summarize_backtest(strategy_df, tau=tau, cost_bps=cost_bps))
    out = pd.DataFrame(rows)
    return out.sort_values(["tau", "cost_bps", "strategy_label"]).reset_index(drop=True)


def build_cumulative_series(all_inputs: pd.DataFrame, tau: float, cost_bps: float, value_col: str) -> pd.DataFrame:
    frames = []
    for strategy_key, strategy_df in all_inputs.groupby("strategy_key", sort=False):
        bt = run_backtest(strategy_df, tau=tau, cost_bps=cost_bps)
        frames.append(
            bt[
                [
                    "date",
                    "strategy_key",
                    "strategy_label",
                    "source_tag",
                    "tau",
                    "cost_bps",
                    value_col,
                    "position",
                    "y_true",
                    "y_pred",
                ]
            ].copy()
        )
    return pd.concat(frames, ignore_index=True)


def plot_cumulative_series(series_df: pd.DataFrame, value_col: str, title: str, output_path: Path, label_order=None):
    plt.figure(figsize=(10, 6))
    if label_order is None:
        label_order = list(series_df["strategy_label"].drop_duplicates())
    for label in label_order:
        subset = series_df[series_df["strategy_label"] == label].sort_values("date")
        if subset.empty:
            continue
        plt.plot(subset["date"], subset[value_col], label=label, linewidth=2)
    plt.axhline(0.0, color="black", linewidth=1, alpha=0.5)
    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Cumulative Return")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()
    print("Saved figure:", output_path)


def write_summary_markdown(main_table: pd.DataFrame, robustness_table: pd.DataFrame, output_path: Path):
    def format_pct(value):
        return "nan" if pd.isna(value) else f"{value * 100:.2f}%"

    def format_sharpe(value):
        return "nan" if pd.isna(value) else f"{value:.3f}"

    lines = [
        "# Strategy Backtest Summary",
        "",
        "## Main Table",
        "",
    ]
    for _, row in main_table.iterrows():
        lines.append(
            "- "
            f"{row['strategy_label']} | cost={row['cost_bps']:.0f}bps | "
            f"net_cum_return={format_pct(row['net_cumulative_return'])} | "
            f"net_sharpe={format_sharpe(row['net_annualized_sharpe'])} | "
            f"net_mdd={format_pct(row['net_max_drawdown'])} | "
            f"turnover={row['turnover']:.0f} | entries={row['entry_count']}"
        )
    lines.extend(
        [
            "",
            "## Two-Stage Robustness",
            "",
        ]
    )
    for _, row in robustness_table.iterrows():
        lines.append(
            "- "
            f"{row['strategy_label']} | cost={row['cost_bps']:.0f}bps | "
            f"net_cum_return={format_pct(row['net_cumulative_return'])} | "
            f"net_sharpe={format_sharpe(row['net_annualized_sharpe'])} | "
            f"net_mdd={format_pct(row['net_max_drawdown'])} | "
            f"turnover={row['turnover']:.0f} | entries={row['entry_count']}"
        )
    output_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print("Saved markdown summary:", output_path)

In [ ]:
mainline_input = build_mainline_strategy_input()
robustness_input = build_two_stage_robustness_input()

mainline_input_path = OUTPUT_DIR / "strategy_input_mainline.csv"
robustness_input_path = OUTPUT_DIR / "strategy_input_two_stage_robustness.csv"

mainline_input.to_csv(mainline_input_path, index=False)
robustness_input.to_csv(robustness_input_path, index=False)

print("Mainline strategy input rows:", len(mainline_input))
print("Two-stage robustness input rows:", len(robustness_input))
display(mainline_input.head())
display(
    mainline_input.groupby("strategy_label")
    .agg(
        n_days=("date", "count"),
        y_true_mean=("y_true", "mean"),
        y_pred_mean=("y_pred", "mean"),
        y_pred_std=("y_pred", "std"),
    )
    .reset_index()
)

In [ ]:
summary_grid = summarize_grid(mainline_input, tau_values=TAU_VALUES, cost_values=COST_BPS_VALUES)
summary_grid["strategy_label"] = pd.Categorical(
    summary_grid["strategy_label"],
    categories=STRATEGY_LABEL_ORDER,
    ordered=True,
)
summary_grid = summary_grid.sort_values(["tau", "cost_bps", "strategy_label"]).reset_index(drop=True)

summary_grid_path = OUTPUT_DIR / "backtest_summary_grid.csv"
summary_grid.to_csv(summary_grid_path, index=False)

main_table = summary_grid[summary_grid["tau"] == MAIN_TAU].copy()
main_table_path = OUTPUT_DIR / "backtest_main_table_tau_5bps.csv"
main_table.to_csv(main_table_path, index=False)

display(
    main_table[
        [
            "strategy_label",
            "cost_bps",
            "net_cumulative_return",
            "net_annualized_sharpe",
            "net_max_drawdown",
            "turnover",
            "entry_count",
            "traded_days",
            "hit_rate_on_traded_days",
        ]
    ]
)

sensitivity_table = summary_grid[
    summary_grid["cost_bps"].eq(5.0)
][
    [
        "strategy_label",
        "tau",
        "cost_bps",
        "net_cumulative_return",
        "net_annualized_sharpe",
        "net_max_drawdown",
        "turnover",
        "entry_count",
        "traded_days",
        "hit_rate_on_traded_days",
    ]
].copy()
sensitivity_table_path = OUTPUT_DIR / "backtest_tau_sensitivity_cost_5bps.csv"
sensitivity_table.to_csv(sensitivity_table_path, index=False)

print("Saved summary grid:", summary_grid_path)
print("Saved main table:", main_table_path)
print("Saved tau sensitivity table:", sensitivity_table_path)

In [ ]:
cumulative_net_main = build_cumulative_series(
    mainline_input,
    tau=MAIN_TAU,
    cost_bps=5.0,
    value_col="net_cum_return",
)
cumulative_gross_main = build_cumulative_series(
    mainline_input,
    tau=MAIN_TAU,
    cost_bps=0.0,
    value_col="gross_cum_return",
)

cumulative_net_main_path = OUTPUT_DIR / "cumulative_returns_net_tau_5bps_cost_5bps.csv"
cumulative_gross_main_path = OUTPUT_DIR / "cumulative_returns_gross_tau_5bps_cost_0bps.csv"
cumulative_net_main.to_csv(cumulative_net_main_path, index=False)
cumulative_gross_main.to_csv(cumulative_gross_main_path, index=False)

plot_cumulative_series(
    cumulative_net_main,
    value_col="net_cum_return",
    title="Strategy Backtest Net Cumulative Return | tau=5bps | cost=5bps",
    output_path=OUTPUT_DIR / "strategy_backtest_net_tau_5bps_cost_5bps.png",
    label_order=STRATEGY_LABEL_ORDER,
)

display(cumulative_net_main.head())

In [ ]:
robustness_summary = summarize_grid(
    robustness_input,
    tau_values=[MAIN_TAU],
    cost_values=[0.0, ROBUSTNESS_COST_BPS],
).sort_values(["cost_bps", "strategy_label"]).reset_index(drop=True)

robustness_summary_path = OUTPUT_DIR / "two_stage_robustness_summary_tau_5bps.csv"
robustness_summary.to_csv(robustness_summary_path, index=False)

robustness_cumulative = build_cumulative_series(
    robustness_input,
    tau=MAIN_TAU,
    cost_bps=ROBUSTNESS_COST_BPS,
    value_col="net_cum_return",
)
robustness_cumulative_path = OUTPUT_DIR / "two_stage_robustness_cumulative_net_tau_5bps_cost_5bps.csv"
robustness_cumulative.to_csv(robustness_cumulative_path, index=False)

plot_cumulative_series(
    robustness_cumulative,
    value_col="net_cum_return",
    title="Two-Stage Robustness | mainline vs dense export v2 | tau=5bps | cost=5bps",
    output_path=OUTPUT_DIR / "two_stage_robustness_net_tau_5bps_cost_5bps.png",
)

display(
    robustness_summary[
        [
            "strategy_label",
            "cost_bps",
            "net_cumulative_return",
            "net_annualized_sharpe",
            "net_max_drawdown",
            "turnover",
            "entry_count",
            "traded_days",
            "hit_rate_on_traded_days",
        ]
    ]
)

write_summary_markdown(
    main_table=main_table[
        [
            "strategy_label",
            "cost_bps",
            "net_cumulative_return",
            "net_annualized_sharpe",
            "net_max_drawdown",
            "turnover",
            "entry_count",
        ]
    ],
    robustness_table=robustness_summary[
        [
            "strategy_label",
            "cost_bps",
            "net_cumulative_return",
            "net_annualized_sharpe",
            "net_max_drawdown",
            "turnover",
            "entry_count",
        ]
    ],
    output_path=OUTPUT_DIR / "strategy_backtest_summary.md",
)

print("Saved robustness summary:", robustness_summary_path)
print("Saved robustness cumulative path:", robustness_cumulative_path)